# Week 14 Optional: Knowledge Distillation

This optional notebook introduces **knowledge distillation** — training a small
"student" model to mimic a large "teacher" model's behavior.

**What we already did (informally):**
In the main notebook, we used Claude Haiku's **hard labels** (fraud/legitimate)
to train DistilBERT. That's a simple form of distillation.

**What formal distillation adds:**
Instead of just using the teacher's final label, we use its **soft probability
distributions** (e.g., "80% fraud, 20% legitimate"). These soft labels contain
richer information — they tell the student *how confident* the teacher was.

**What you'll learn:**
1. Hard labels vs soft labels (temperature scaling)
2. Distillation loss (KL divergence between teacher and student distributions)
3. Train a student model with combined task loss + distillation loss
4. Compare: hard-label training vs soft-label distillation

**Prerequisites**: Completed Week 14 main notebook

**GPU**: T4 GPU recommended (Runtime → Change runtime type → T4 GPU)

In [ ]:
# =============================================================================
# SETUP: Install and Import Libraries
# =============================================================================

!pip install -q transformers datasets evaluate scikit-learn

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from datasets import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import evaluate
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Device check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\n✅ Setup complete!")

# Section 1: Hard Labels vs Soft Labels

## What We Did in the Main Notebook

In the main notebook, we trained DistilBERT on **hard labels** — each transaction
was labeled as either "fraud" (1) or "legitimate" (0). No ambiguity.

## What Soft Labels Add

A teacher model doesn't just output "fraud" — it outputs a **probability distribution**:
- "92% fraud, 8% legitimate" → the teacher is very confident
- "55% fraud, 45% legitimate" → the teacher is uncertain, this is an edge case

**Soft labels preserve this uncertainty.** They tell the student:
- "This transaction is clearly fraud" (high confidence)
- "This one is borderline — could go either way" (low confidence)

This is richer information than a hard 0/1 label!

## Temperature Scaling

To make soft labels even softer (more informative), we use **temperature scaling**:

```python
# Standard softmax (T=1): sharp distribution
softmax([3.0, 1.0])  # → [0.88, 0.12]

# High temperature (T=3): smoother distribution
softmax([3.0/3, 1.0/3])  # → [0.66, 0.34]
```

Higher temperature → smoother distribution → more information transferred.
The standard distillation temperature is T=2 to T=4.

In [ ]:
# =============================================================================
# DEMO: Visualizing Temperature Scaling
# =============================================================================

# Raw logits from a model (before softmax)
logits = torch.tensor([3.0, 1.0])

temperatures = [0.5, 1.0, 2.0, 4.0, 8.0]
fig, axes = plt.subplots(1, len(temperatures), figsize=(15, 3))

for ax, T in zip(axes, temperatures):
    probs = F.softmax(logits / T, dim=0).numpy()
    ax.bar(['fraud', 'legit'], probs, color=['#e74c3c', '#2ecc71'])
    ax.set_title(f'T = {T}')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Probability')
    for i, p in enumerate(probs):
        ax.text(i, p + 0.02, f'{p:.2f}', ha='center', fontsize=9)

plt.suptitle('Effect of Temperature on Softmax Distribution', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("T=0.5: Very sharp (almost one-hot) — minimal information transfer")
print("T=1.0: Standard softmax — what models normally output")
print("T=2-4: Sweet spot for distillation — preserves uncertainty")
print("T=8.0: Very flat — almost uniform, too much smoothing")

In [ ]:
# =============================================================================
# DEMO: Prepare Data and Train Teacher Model
# =============================================================================

MODEL_NAME = "distilbert-base-uncased"
ID2LABEL = {0: "legitimate", 1: "fraud"}
LABEL2ID = {"legitimate": 0, "fraud": 1}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load or generate training data (same as main notebook)
import os

CSV_PATH = "synthetic_fraud_data.csv"

if os.path.exists(CSV_PATH):
    train_df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(train_df)} synthetic transactions from Week 13")
else:
    print("Week 13 CSV not found — using template-based training data")
    import random
    random.seed(42)
    fraud_templates = [
        "Unauthorized wire transfer of ${amount} to unknown account in {country}",
        "Multiple rapid ATM withdrawals totaling ${amount} across {n} locations",
        "Online purchase of ${amount} from suspicious merchant, shipping to {country}",
        "Account takeover: password changed and ${amount} transferred within minutes",
        "Card-not-present transaction of ${amount} from unrecognized IP address",
    ]
    legit_templates = [
        "Monthly payroll deposit of ${amount} from registered employer",
        "Grocery purchase of ${amount} at local supermarket",
        "Recurring utility payment of ${amount} to electric company",
        "ATM withdrawal of ${amount} at usual branch location",
        "Online subscription renewal of ${amount} for streaming service",
    ]
    rows = []
    for i in range(50):
        t = random.choice(fraud_templates)
        desc = t.replace("${amount}", str(random.randint(500, 50000)))
        desc = desc.replace("{country}", random.choice(["Nigeria", "Romania", "Russia"]))
        desc = desc.replace("{n}", str(random.randint(3, 8)))
        rows.append({"description": desc, "label": "fraud"})
    for i in range(50):
        t = random.choice(legit_templates)
        desc = t.replace("${amount}", str(random.randint(20, 5000)))
        rows.append({"description": desc, "label": "legitimate"})
    train_df = pd.DataFrame(rows)

# Split data
train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label'])

def tokenize_fn(examples):
    return tokenizer(examples['description'], truncation=True, max_length=128)

def make_dataset(df):
    ds = Dataset.from_pandas(df.assign(label=df['label'].map(LABEL2ID))[['description', 'label']])
    ds = ds.map(tokenize_fn, batched=True)
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    return ds

train_ds = make_dataset(train_split)
val_ds = make_dataset(val_split)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

# --- Train Teacher Model (full fine-tuning, 3 epochs) ---
print("\n🎓 Training TEACHER model (full fine-tuning)...")
teacher_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    acc = accuracy_score(eval_pred.label_ids, preds)
    _, _, f1, _ = precision_recall_fscore_support(eval_pred.label_ids, preds, average='binary')
    return {'accuracy': acc, 'f1': f1}

teacher_trainer = Trainer(
    model=teacher_model,
    args=TrainingArguments(
        output_dir='./teacher-model', num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        eval_strategy='epoch', learning_rate=2e-5, weight_decay=0.01,
        logging_steps=50, report_to='none', fp16=torch.cuda.is_available(),
    ),
    train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)
teacher_trainer.train()

teacher_eval = teacher_trainer.evaluate()
print(f"\n✅ Teacher trained! Val accuracy: {teacher_eval['eval_accuracy']:.1%}")

In [ ]:
# =============================================================================
# DEMO: Generate Soft Labels from Teacher
# =============================================================================
# Get the teacher's probability distributions for every training example.

teacher_model.eval()

# Get teacher predictions (logits) for all training data
teacher_preds = teacher_trainer.predict(train_ds)
teacher_logits = torch.tensor(teacher_preds.predictions)

# Show examples of hard vs soft labels
print("Hard Labels vs Soft Labels (Teacher Probabilities):")
print("=" * 60)

TEMPERATURE = 3.0

for i in range(5):
    hard_label = train_ds[i]['label'].item()
    logits_i = teacher_logits[i]

    # Soft labels at T=1 (standard)
    soft_t1 = F.softmax(logits_i, dim=0).numpy()

    # Soft labels at T=3 (distillation temperature)
    soft_t3 = F.softmax(logits_i / TEMPERATURE, dim=0).numpy()

    desc = train_split.iloc[i]['description'][:50]
    print(f"\n  \"{desc}...\"")
    print(f"    Hard label:  {ID2LABEL[hard_label]}")
    print(f"    Soft (T=1):  legit={soft_t1[0]:.3f}, fraud={soft_t1[1]:.3f}")
    print(f"    Soft (T=3):  legit={soft_t3[0]:.3f}, fraud={soft_t3[1]:.3f}")

print(f"\nNotice how T={TEMPERATURE} spreads the probabilities — even confident")
print("predictions now share some probability mass with the other class.")

# Section 2: The Distillation Loss

## Combined Loss Function

The distillation loss combines two objectives:

1. **Task Loss** (standard cross-entropy): Student learns from hard labels
2. **Distillation Loss** (KL divergence): Student mimics teacher's soft distributions

```python
total_loss = alpha * task_loss + (1 - alpha) * distillation_loss
```

Where:
- `alpha` controls the balance (typically 0.5)
- `task_loss = CrossEntropy(student_logits, hard_labels)`
- `distillation_loss = KL_Divergence(student_soft, teacher_soft) * T^2`

The `T^2` factor compensates for the temperature scaling — without it, the
gradients from the soft labels would be too small.

## Why KL Divergence?

KL divergence measures how different two probability distributions are:
- KL(teacher || student) = 0 → student perfectly matches teacher
- Higher KL → student's distribution differs from teacher's

It's the natural loss for matching probability distributions.

In [ ]:
# =============================================================================
# DEMO: Custom Distillation Trainer
# =============================================================================

class DistillationTrainer(Trainer):
    """Custom Trainer that adds distillation loss to standard training."""

    def __init__(self, teacher_logits, temperature=3.0, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.teacher_logits = teacher_logits.to(device)
        self.temperature = temperature
        self.alpha = alpha  # Weight for task loss vs distillation loss

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Get hard labels and remove from inputs
        labels = inputs.pop("labels")

        # Forward pass through student
        outputs = model(**inputs)
        student_logits = outputs.logits

        # 1. Standard task loss (cross-entropy with hard labels)
        task_loss = F.cross_entropy(student_logits, labels)

        # 2. Distillation loss (KL divergence with soft labels)
        # Get teacher soft labels for this batch
        batch_indices = list(range(
            self.state.global_step * self.args.per_device_train_batch_size,
            min(
                (self.state.global_step + 1) * self.args.per_device_train_batch_size,
                len(self.teacher_logits)
            )
        ))

        # Handle batch size mismatch at end of epoch
        if len(batch_indices) != student_logits.shape[0]:
            batch_indices = batch_indices[:student_logits.shape[0]]

        teacher_batch = self.teacher_logits[batch_indices]

        # Compute soft distributions at temperature T
        student_soft = F.log_softmax(student_logits / self.temperature, dim=-1)
        teacher_soft = F.softmax(teacher_batch / self.temperature, dim=-1)

        # KL divergence loss, scaled by T^2
        distill_loss = F.kl_div(
            student_soft, teacher_soft, reduction='batchmean'
        ) * (self.temperature ** 2)

        # Combined loss
        loss = self.alpha * task_loss + (1 - self.alpha) * distill_loss

        return (loss, outputs) if return_outputs else loss

print("DistillationTrainer defined!")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Alpha: 0.5 (equal weight task + distillation)")
print(f"  Teacher logits shape: {teacher_logits.shape}")

In [ ]:
# =============================================================================
# DEMO: Train Student with Distillation
# =============================================================================

# Fresh student model
student_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
)

# Train with distillation
distill_trainer = DistillationTrainer(
    teacher_logits=teacher_logits,
    temperature=TEMPERATURE,
    alpha=0.5,
    model=student_model,
    args=TrainingArguments(
        output_dir='./student-distilled',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy='epoch',
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=10,
        report_to='none',
        fp16=torch.cuda.is_available(),
    ),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Training STUDENT with distillation (soft labels from teacher)...")
print("=" * 50)
distill_result = distill_trainer.train()
print("=" * 50)

distill_eval = distill_trainer.evaluate()
print(f"\n✅ Distilled student trained!")
print(f"  Val accuracy: {distill_eval['eval_accuracy']:.1%}")
print(f"  Val F1:       {distill_eval['eval_f1']:.3f}")

In [ ]:
# =============================================================================
# DEMO: Compare Hard-Label vs Distilled Student
# =============================================================================
# Train another student with just hard labels (no distillation) for comparison.

hard_student = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
)

hard_trainer = Trainer(
    model=hard_student,
    args=TrainingArguments(
        output_dir='./student-hard-labels',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy='epoch',
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=50,
        report_to='none',
        fp16=torch.cuda.is_available(),
    ),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Training baseline student (hard labels only, no distillation)...")
hard_trainer.train()
hard_eval = hard_trainer.evaluate()

# Comparison
print("\n" + "=" * 60)
print("COMPARISON: Hard Labels vs Distillation")
print("=" * 60)

comparison = pd.DataFrame([
    {
        'Method': 'Teacher (full fine-tuned)',
        'Val Accuracy': f"{teacher_eval['eval_accuracy']:.1%}",
        'Val F1': f"{teacher_eval['eval_f1']:.3f}",
        'Note': 'Upper bound',
    },
    {
        'Method': 'Student (hard labels only)',
        'Val Accuracy': f"{hard_eval['eval_accuracy']:.1%}",
        'Val F1': f"{hard_eval['eval_f1']:.3f}",
        'Note': 'What main notebook does',
    },
    {
        'Method': f'Student (distilled, T={TEMPERATURE})',
        'Val Accuracy': f"{distill_eval['eval_accuracy']:.1%}",
        'Val F1': f"{distill_eval['eval_f1']:.3f}",
        'Note': 'Soft labels from teacher',
    },
])
display(comparison)

print("\nDistillation often helps most when:")
print("  - The dataset is small (soft labels add information)")
print("  - The teacher is significantly larger than the student")
print("  - There are ambiguous/borderline examples")

In [ ]:
# =============================================================================
# DEMO: Visualizing What Distillation Transfers
# =============================================================================
# Let's look at HOW the distilled student's predictions differ from hard-label.

# Get predictions from both students on validation set
distill_preds = distill_trainer.predict(val_ds)
hard_preds = hard_trainer.predict(val_ds)

# Compare confidence distributions
distill_probs = F.softmax(torch.tensor(distill_preds.predictions), dim=-1).numpy()
hard_probs = F.softmax(torch.tensor(hard_preds.predictions), dim=-1).numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Confidence histogram — hard label student
hard_max_conf = hard_probs.max(axis=1)
ax1.hist(hard_max_conf, bins=20, color='#3498db', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Max Prediction Confidence')
ax1.set_ylabel('Count')
ax1.set_title('Hard-Label Student')
ax1.axvline(0.5, color='red', linestyle='--', alpha=0.5)

# Confidence histogram — distilled student
distill_max_conf = distill_probs.max(axis=1)
ax2.hist(distill_max_conf, bins=20, color='#e74c3c', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Max Prediction Confidence')
ax2.set_ylabel('Count')
ax2.set_title('Distilled Student')
ax2.axvline(0.5, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Prediction Confidence Distribution', fontsize=14)
plt.tight_layout()
plt.show()

print("Key observation: Distilled students often have better-calibrated")
print("confidence — they're less overconfident on uncertain examples.")

## Lab: Temperature Sweep for Distillation

### Your Task

Experiment with different temperatures (T=1, 2, 4, 8) and alphas (0.3, 0.5, 0.7)
to find the best distillation configuration.

### Steps

1. Loop through temperatures [1, 2, 4, 8]
2. For each temperature, train a distilled student (1 epoch for speed)
3. Record validation accuracy and F1
4. Create a summary table and identify the best temperature
5. (Bonus) Try different alpha values with your best temperature

### Expected Output

- Table with 4 rows (one per temperature)
- Best temperature identified
- Comparison to hard-label baseline

### Homework Extension

Try distilling from a *larger* teacher (e.g., `bert-base-uncased`, 110M params)
to DistilBERT (67M params). Does the larger teacher gap make distillation
more beneficial?

In [ ]:
# =============================================================================
# SOLUTION: LAB — TEMPERATURE SWEEP FOR DISTILLATION
# =============================================================================

temperatures_to_try = [1, 2, 4, 8]
lab_results = []

for T in temperatures_to_try:
    print(f"\n{'='*40}")
    print(f"Distilling with T={T}")
    print(f"{'='*40}")

    # Create a fresh student model
    student = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
    )

    # Create DistillationTrainer with this temperature
    lab_trainer = DistillationTrainer(
        teacher_logits=teacher_logits,
        temperature=T,
        alpha=0.5,
        model=student,
        args=TrainingArguments(
            output_dir=f'./student-T{T}',
            num_train_epochs=1,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            eval_strategy='epoch',
            learning_rate=2e-5,
            weight_decay=0.01,
            logging_steps=50,
            report_to='none',
            fp16=torch.cuda.is_available(),
        ),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # Train for 1 epoch
    lab_trainer.train()

    # Evaluate
    eval_result = lab_trainer.evaluate()
    val_acc = eval_result['eval_accuracy']
    val_f1 = eval_result['eval_f1']

    lab_results.append({
        'temperature': T,
        'val_accuracy': val_acc,
        'val_f1': val_f1,
    })
    print(f"  Accuracy: {val_acc:.1%}, F1: {val_f1:.3f}")

# Summary table
temp_df = pd.DataFrame(lab_results)
print(f"\n{'='*60}")
print("Temperature Experiment Summary:")
display(temp_df)

best = temp_df.loc[temp_df['val_accuracy'].idxmax()]
print(f"\nBest temperature: T={best['temperature']}, accuracy={best['val_accuracy']:.1%}")
print(f"Hard-label baseline: {hard_eval['eval_accuracy']:.1%}")
print("\nLab complete!")

# Summary

## What You Learned

| Concept | Description |
|---------|-------------|
| **Hard labels** | Binary 0/1 labels — what we used in the main notebook |
| **Soft labels** | Teacher's probability distributions — richer information |
| **Temperature** | Controls distribution smoothness (T=2-4 typical) |
| **KL Divergence** | Loss function for matching distributions |
| **Alpha** | Balances task loss vs distillation loss |

## Key Takeaways

1. **Distillation transfers "dark knowledge"** — the teacher's uncertainty about borderline cases
2. **Temperature scaling** makes soft labels more informative by smoothing distributions
3. **Combined loss** (alpha * task + (1-alpha) * distillation) balances learning from labels and teacher
4. **Best gains** come when: small datasets, large teacher-student gap, ambiguous examples
5. **What we did in Week 13-14** (LLM generates labels → train small model) is informal distillation!

## The Distillation Spectrum

| Approach | Teacher | Student | Labels Used |
|----------|---------|---------|-------------|
| **Week 14 main notebook** | Claude Haiku (LLM) | DistilBERT | Hard (fraud/legit) |
| **This notebook** | Fine-tuned DistilBERT | Fresh DistilBERT | Soft (probabilities) |
| **Production** | GPT-4 / Claude | Small BERT | Soft + hard combined |

## Resources

- [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531) — Hinton et al., 2015
- [HuggingFace Distillation Guide](https://huggingface.co/docs/transformers/tasks/knowledge_distillation)
- [DistilBERT Paper](https://arxiv.org/abs/1910.01108) — itself a product of distillation!